# Benchmark de NLP Financeiro — tradução, representações e sentimento

**TF-IDF + Regressão Logística · BiLSTM + Optuna · BERTimbau + LightGBM · BERTimbau + LoRA**

Este notebook é simultaneamente uma **aula prática de Processamento de Linguagem Natural** e um benchmark controlado. Começaremos pelo objeto mais simples — uma sequência de caracteres — e avançaremos até representações esparsas, embeddings aprendidos, atenção e adaptação eficiente de Transformers.

O problema ponta a ponta é:

$$
x_{\mathrm{en}}
\xrightarrow{\text{tradução}}
\hat{x}_{\mathrm{pt}}
\xrightarrow{\text{classificador}}
\hat{y}\in\{\text{negative},\text{neutral},\text{positive}\}.
$$

Usaremos a tradução automática `text_pt_mt` como entrada principal e a tradução portuguesa de referência `text_pt` como **controle experimental**. A diferença entre elas ajuda a separar falhas do tradutor de falhas do classificador.

> **Regra central:** treino e validação escolhem; o teste apenas mede a solução já escolhida.

## Como usar este notebook

1. Execute primeiro toda a **Parte I**. Ela apresenta os fundamentos, valida os dados e materializa um único split estratificado.
2. Informe `DATA_PATH` na célula de configuração.
3. Ative `EXECUTAR_TRADUCAO=True` somente quando quiser gerar o cache de traduções.
4. Execute os experimentos individualmente pelas flags `EXECUTAR_*`.
5. Para uma execução rápida de depuração, use `MODO_RAPIDO=True`.
6. Nunca altere o split entre modelos. Todos leem o mesmo arquivo `data/splits_nlp_seed42.parquet`.

Os blocos custosos vêm desligados por padrão para evitar iniciar treinamento ou downloads acidentalmente. Isso não reduz a completude do código: basta ativar a respectiva flag.

### Contrato científico

- `SEED = 42`;
- F1 Macro é a métrica principal;
- nenhum ajuste usa o teste;
- a tradução não usa `y`;
- todos os classificadores recebem exatamente os mesmos exemplos;
- tempos de tradução, treinamento e inferência são medidos separadamente;
- artefatos intermediários são persistidos para reprodutibilidade.

# Parte I — Fundamentos e exploração guiada

## 1. Do texto à decisão

**Pergunta de aprendizagem:** como uma frase financeira vira números e, depois, uma classe?

Computadores não recebem “significado” diretamente. Um pipeline de NLP transforma texto em estruturas numéricas:

1. caracteres Unicode formam uma string;
2. um tokenizador define unidades como palavras ou subpalavras;
3. uma representação converte tokens em vetores;
4. um classificador produz **logits**;
5. softmax converte logits em probabilidades;
6. `argmax` escolhe a classe.

<p align="center">
  <img src="./NLP_PIPELINE_FINANCEIRO.png" width="100%">
</p>

O infográfico mostra quatro caminhos de representação. Eles compartilham o mesmo alvo e o mesmo protocolo, mas carregam hipóteses diferentes sobre como codificar linguagem.

## 2. O que é texto para o computador?

Uma string é uma sequência de **pontos de código Unicode**, não uma sequência de palavras. A frase:

```text
A receita cresceu 12,5%.
```

contém letras, espaços, pontuação e símbolos. Antes de modelar, precisamos decidir o que preservar. Em finanças, números, sinais, moedas, percentuais e negações frequentemente carregam informação preditiva.

### Normalização não é sinônimo de “apagar”

Uma limpeza agressiva poderia transformar:

```text
O lucro não cresceu 10%.
```

em algo próximo de:

```text
lucro cresceu
```

Isso inverte ou empobrece o significado. Por esse motivo, nossa limpeza clássica será moderada; para BERTimbau, preservaremos o texto original.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. `frase_exemplo` é uma string Python; internamente, seus elementos são caracteres Unicode.
# 2. `ord` revela o ponto de código e `encode` mostra sua representação física em UTF-8.
# 3. A codificação em bytes interessa ao armazenamento; o modelo recebe tokens/IDs, não esses bytes.
# 4. `split()` separa apenas em espaços e mantém vírgulas junto às palavras.
# 5. A célula não modifica dados nem aprende parâmetros: é uma demonstração conceitual isolada.
# Saída esperada: string, contagem, pares caractere-código, bytes e lista de tokens ingênuos.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Inspecionar caracteres, bytes UTF-8 e uma tokenização simples sem bibliotecas.
# -----------------------------------------------------------------------------
frase_exemplo = "O lucro não cresceu 12,5%, apesar da receita maior."

print("String:", frase_exemplo)
print("Número de caracteres:", len(frase_exemplo))
print("Primeiros pontos de código:", [(c, ord(c)) for c in frase_exemplo[:12]])
print("UTF-8:", frase_exemplo.encode("utf-8")[:30])

# split() é útil didaticamente, mas não é um tokenizador linguístico completo:
# pontuação e palavras continuam misturadas.
tokens_ingenuos = frase_exemplo.split()
print("Tokenização ingênua:", tokens_ingenuos)

## 3. Tokenização: palavras, vocabulário e subpalavras

Um tokenizador implementa uma função

$$
T(x)=(t_1,t_2,\ldots,t_L),
$$

onde $L$ é o comprimento da sequência. Há duas estratégias importantes neste projeto:

- **palavras** na BiLSTM: vocabulário criado apenas com o treino; itens desconhecidos viram `<UNK>`;
- **subpalavras** no BERTimbau: vocabulário pré-treinado que pode decompor palavras raras.

Para processar várias frases em paralelo, usamos *padding* até um comprimento comum. A máscara de atenção indica quais posições são reais:

$$
m_i =
\begin{cases}
1,& \text{token real}\\
0,& \text{padding}.
\end{cases}
$$

O vocabulário e qualquer estatística de normalização devem ser ajustados **somente no treino**.

### Tokenização em detalhes: do texto ao tensor

Considere um batch com $N$ frases. Depois da tokenização, cada frase possui comprimento próprio $L_n$. Redes neurais trabalham melhor com tensores retangulares; por isso, completamos as sequências até

$$
L_{\max}=\max_{n\in\{1,\ldots,N\}}L_n.
$$

O batch de IDs passa a ter forma:

$$
X_{\mathrm{ids}}\in\mathbb{N}^{N\times L_{\max}}.
$$

Exemplo:

```text
"lucro cresceu"        → [  91, 8301,    0,    0]
"receita não cresceu"  → [2450,  346, 8301,    0]
"margens diminuíram"   → [7211, 9182,    0,    0]
                          └───────────────┘
                           Lmax = 4
```

O zero representa `<PAD>`, não uma palavra real. Para Transformers, uma máscara paralela informa:

```text
input_ids:      [2450, 346, 8301, 0]
attention_mask: [   1,   1,    1, 0]
```

#### Vocabulário da BiLSTM

O vocabulário próprio é um dicionário:

$$
\mathcal{V}: \text{token}\mapsto\text{índice}.
$$

Somente o treino pode definir $\mathcal{V}$. Se usássemos validação ou teste, até uma operação “não supervisionada” revelaria quais palavras aparecerão no futuro. Termos não vistos recebem `<UNK>`.

#### Subpalavras do BERTimbau

BERTimbau usa um vocabulário fixo aprendido no pré-treinamento. Uma palavra rara pode ser dividida:

```text
"descapitalização" → ["des", "##capital", "##ização"]
```

Os marcadores exatos dependem do tokenizer. A vantagem é reduzir palavras totalmente desconhecidas; a desvantagem é que o comprimento em tokens não coincide com o número de palavras. `MAX_LENGTH_BERT` limita **subpalavras**, não palavras.

| Conceito | BiLSTM | BERTimbau |
|---|---|---|
| Unidade | palavra/pontuação por regex | subpalavra do checkpoint |
| Vocabulário | aprendido no treino | herdado do pré-treinamento |
| Desconhecidos | `<UNK>` | decomposição em subpalavras |
| Padding | `padding_idx=0` | tokenizer + `attention_mask` |
| Truncamento | opcional no Dataset | `max_length` explícito |

## 4. Representações esparsas: Bag of Words e TF-IDF

O modelo clássico ignora a maior parte da ordem global e representa documentos pela presença ou frequência de termos. Para o termo $t$ no documento $d$:

$$
\operatorname{tfidf}(t,d)
=\operatorname{tf}(t,d)\times
\log\left(\frac{1+N}{1+\operatorname{df}(t)}\right)+1.
$$

- $\operatorname{tf}(t,d)$: frequência do termo no documento;
- $\operatorname{df}(t)$: número de documentos que contêm o termo;
- $N$: total de documentos de treino.

Bigramas preservam pequenos fragmentos de ordem, como `não cresceu`. O resultado é um vetor de alta dimensionalidade com muitos zeros.

A Regressão Logística multiclasse calcula:

$$
\mathbf{z}=W\mathbf{x}+\mathbf{b},
\qquad
p(y=k\mid x)=\frac{e^{z_k}}{\sum_j e^{z_j}}.
$$

### O que o TF-IDF realmente aprende?

O TF-IDF possui duas etapas distintas:

1. **ajuste do vocabulário e do IDF**, feito apenas no treino;
2. **transformação**, aplicada a treino, validação e teste com o vocabulário já congelado.

Se $\mathbf{x}_d\in\mathbb{R}^{V}$ representa o documento $d$, então $V$ é o número de unigramas e bigramas retidos. Em geral, $\mathbf{x}_d$ é esparso:

$$
\operatorname{sparsity}
=1-\frac{\#\{x_{d,j}\neq0\}}{N\times V}.
$$

`min_df=2` remove termos que aparecem em apenas um documento de treino, reduzindo ruído. `sublinear_tf=True` substitui a frequência bruta por:

$$
\operatorname{tf}_{sublinear}(t,d)=1+\log(\operatorname{tf}(t,d)),
$$

quando a frequência é positiva. Assim, repetir uma palavra dez vezes não gera dez vezes mais evidência.

Na regressão logística, cada classe $k$ possui um vetor de pesos $w_k$. O logit é:

$$
z_k=w_k^\top x+b_k.
$$

Um coeficiente positivo alto para o bigrama `lucro aumentou` eleva o logit da classe correspondente. Entretanto, coeficiente não prova causalidade: ele reflete correlações condicionadas ao corpus, ao vocabulário e à regularização.

A regularização L2 penaliza pesos grandes:

$$
\mathcal{J}(W)
=\mathcal{L}_{CE}(W)+\lambda\lVert W\rVert_2^2.
$$

No scikit-learn, `C` é aproximadamente o inverso da força de regularização: `C` menor implica regularização mais forte.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. Criamos três documentos pequenos para enxergar matrizes que seriam enormes no corpus real.
# 2. `CountVectorizer.fit_transform` aprende o vocabulário e conta unigramas/bigramas.
# 3. `.toarray()` é seguro apenas neste exemplo; no benchmark preservaríamos a matriz esparsa.
# 4. O segundo vetorizador recalcula os mesmos tipos de termos, agora ponderados por TF-IDF.
# 5. `get_feature_names_out` mantém a correspondência exata entre coluna e termo.
# 6. Os DataFrames são apenas visualizações; não participam do treino final.
# Saída esperada: uma matriz de contagens e outra de pesos TF-IDF.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Visualizar, em um microcorpus, como contagens e TF-IDF viram matrizes.
# Esta demonstração é independente do dataset principal.
# -----------------------------------------------------------------------------
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd

microcorpus = [
    "lucro cresceu fortemente",
    "lucro não cresceu",
    "receita cresceu",
]

cv_demo = CountVectorizer(ngram_range=(1, 2))
X_count_demo = cv_demo.fit_transform(microcorpus)
display(pd.DataFrame(
    X_count_demo.toarray(),
    columns=cv_demo.get_feature_names_out(),
    index=[f"d{i}" for i in range(len(microcorpus))]
))

tfidf_demo = TfidfVectorizer(ngram_range=(1, 2))
X_tfidf_demo = tfidf_demo.fit_transform(microcorpus)
display(pd.DataFrame(
    X_tfidf_demo.toarray().round(3),
    columns=tfidf_demo.get_feature_names_out(),
    index=[f"d{i}" for i in range(len(microcorpus))]
))

## 5. Embeddings e BiLSTM

Um embedding associa cada token $w$ a um vetor denso:

$$
E[w]\in\mathbb{R}^{d}.
$$

Na BiLSTM, os embeddings são aprendidos do zero. Uma LSTM controla memória por portas. Em forma compacta:

$$
\begin{aligned}
i_t &= \sigma(W_i x_t + U_i h_{t-1}+b_i),\\
f_t &= \sigma(W_f x_t + U_f h_{t-1}+b_f),\\
o_t &= \sigma(W_o x_t + U_o h_{t-1}+b_o),\\
\tilde{c}_t &= \tanh(W_c x_t + U_c h_{t-1}+b_c),\\
c_t &= f_t\odot c_{t-1}+i_t\odot\tilde{c}_t,\\
h_t &= o_t\odot\tanh(c_t).
\end{aligned}
$$

A BiLSTM lê a sequência nos dois sentidos. Concatenamos os estados ocultos finais:

$$
h=[\overrightarrow{h}_L;\overleftarrow{h}_1],
$$

aplicamos dropout e uma camada linear. Diferente do TF-IDF, a ordem influencia explicitamente a representação.

## 6. Atenção, BERTimbau e LoRA

Em self-attention, cada token produz *queries*, *keys* e *values*:

$$
Q=XW_Q,\quad K=XW_K,\quad V=XW_V,
$$

e a atenção é:

$$
\operatorname{Attention}(Q,K,V)
=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V.
$$

BERTimbau já foi pré-treinado em grande corpus em português. Podemos:

- congelá-lo e extrair embeddings para LightGBM;
- adaptá-lo com LoRA.

LoRA mantém a matriz original $W_0$ congelada e aprende uma atualização de baixo posto:

$$
W=W_0+\Delta W,\qquad
\Delta W=\frac{\alpha}{r}BA,
$$

com $A\in\mathbb{R}^{r\times d}$, $B\in\mathbb{R}^{d\times r}$ e $r\ll d$.

<p align="center">
  <img src="./BERTIMBAU_LORA.png" width="100%">
</p>

## 7. Logits, probabilidades, perda e F1 Macro

Os modelos produzem três logits $\mathbf{z}=[z_0,z_1,z_2]$. Eles não são probabilidades. Softmax produz:

$$
p_k=\frac{e^{z_k}}{\sum_j e^{z_j}},\qquad \sum_k p_k=1.
$$

Para um exemplo cuja classe verdadeira é $y$, a entropia cruzada é:

$$
\mathcal{L}_{CE}=-\log p_y.
$$

Para cada classe $k$:

$$
P_k=\frac{TP_k}{TP_k+FP_k},\quad
R_k=\frac{TP_k}{TP_k+FN_k},\quad
F1_k=\frac{2P_kR_k}{P_k+R_k}.
$$

A métrica principal dá o mesmo peso a cada classe:

$$
F1_{\mathrm{macro}}=\frac{1}{K}\sum_{k=1}^{K}F1_k.
$$

Isso é importante quando `neutral` é mais frequente que as demais classes.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. Os três números de `logits_demo` são escores livres, um por classe.
# 2. Subtrair o máximo não altera a softmax, mas evita exponenciais numericamente gigantes.
# 3. Dividir cada exponencial pela soma cria uma distribuição cuja soma é um.
# 4. `argmax` retorna o índice da maior probabilidade; a lista converte índice em nome.
# 5. Durante treinamento, CrossEntropyLoss faz a versão estável desse cálculo internamente.
# Saída esperada: probabilidades válidas e a classe associada ao maior logit.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Converter logits em probabilidades e confirmar a soma unitária.
# -----------------------------------------------------------------------------
import numpy as np

logits_demo = np.array([-0.4, 0.2, 1.6], dtype=float)
exp_estavel = np.exp(logits_demo - logits_demo.max())  # evita overflow
probs_demo = exp_estavel / exp_estavel.sum()

print("Logits:", logits_demo)
print("Probabilidades:", probs_demo.round(4))
print("Soma:", probs_demo.sum().round(6))
print("Classe prevista:", ["negative", "neutral", "positive"][probs_demo.argmax()])

### Checkpoint conceitual 1

- Texto precisa ser representado numericamente, mas nenhuma representação é neutra.
- TF-IDF enfatiza frequência discriminativa e oferece interpretabilidade.
- BiLSTM aprende embeddings e ordem a partir do corpus supervisionado.
- BERTimbau transfere conhecimento linguístico prévio.
- LoRA adapta apenas uma pequena fração dos parâmetros.
- Softmax não “cria” evidência; apenas normaliza os logits.
- F1 Macro impede que uma classe majoritária domine a avaliação.

## 8. Bibliotecas, ambiente e dependências

As dependências são separadas em dois grupos:

- básicas: NumPy, pandas, matplotlib, seaborn, scikit-learn e PyTorch;
- experimentais: Optuna, LightGBM, Transformers, Datasets, Accelerate, PEFT e SentencePiece.

A célula abaixo **diagnostica** as bibliotecas opcionais. Ela não instala silenciosamente pacotes no ambiente.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. `importlib.import_module` tenta carregar cada dependência no kernel atual.
# 2. O diagnóstico separa ausência de pacote de erros de importação e registra versões.
# 3. Bibliotecas opcionais não são instaladas automaticamente, preservando o ambiente do usuário.
# 4. Versões são importantes porque APIs de Transformers e PEFT evoluem.
# 5. Sistema e Python completam o contexto necessário para reproduzir um experimento.
# Saída esperada: tabela de disponibilidade e identificação do ambiente.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Diagnosticar versões e dependências antes de executar os experimentos.
# -----------------------------------------------------------------------------
import importlib
import platform
import sys

pacotes = [
    "numpy", "pandas", "sklearn", "torch", "matplotlib", "seaborn",
    "optuna", "lightgbm", "transformers", "datasets", "accelerate",
    "peft", "sentencepiece", "pyarrow"
]

diagnostico = []
for nome in pacotes:
    try:
        modulo = importlib.import_module(nome)
        diagnostico.append((nome, True, getattr(modulo, "__version__", "sem __version__")))
    except Exception as erro:
        diagnostico.append((nome, False, type(erro).__name__))

display(pd.DataFrame(diagnostico, columns=["pacote", "disponível", "versão/erro"]))
print("Python:", sys.version.split()[0])
print("Sistema:", platform.platform())

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. Importamos apenas utilitários compartilhados por várias seções do notebook.
# 2. `SEED` sincroniza Python, NumPy, CPU e todas as GPUs do PyTorch.
# 3. `PYTHONHASHSEED` ajuda a estabilizar estruturas dependentes de hash.
# 4. `deterministic=True` prioriza repetibilidade; pode reduzir desempenho em CUDA.
# 5. `benchmark=False` evita que a cuDNN escolha algoritmos com variação entre execuções.
# 6. `DEVICE` centraliza a escolha CPU/GPU para não espalhar condicionais pelo código.
# Saída esperada: dispositivo selecionado e estado aleatório inicializado.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Importar bibliotecas comuns e fixar as fontes de aleatoriedade.
# -----------------------------------------------------------------------------
import json
import os
import random
import re
import time
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    log_loss, precision_score, recall_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

## 9. Configuração central e controles de custo

Todos os caminhos, modelos e flags ficam em um único lugar. Isso reduz estados ocultos e torna claro o que será executado.

O modelo de tradução sugerido é `Helsinki-NLP/opus-mt-tc-big-en-pt`. O backbone comum dos dois experimentos contextuais é `neuralmind/bert-base-portuguese-cased`.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. Esta é a única célula que o leitor deve editar para caminhos e orçamento computacional.
# 2. As pastas são criadas antes de qualquer gravação para evitar falhas tardias.
# 3. `LABEL2ID` e `ID2LABEL` fixam uma ordem única para métricas, logits e relatórios.
# 4. Comprimentos máximos limitam memória e tornam truncamento uma decisão explícita.
# 5. Flags impedem downloads/treinos custosos acidentais; `MODO_RAPIDO` serve para depuração.
# 6. Mudar uma flag não muda os splits, pois eles são persistidos em arquivo próprio.
# Saída esperada: configuração completa impressa antes dos experimentos.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Centralizar caminhos, modelos, limites e flags de execução.
# -----------------------------------------------------------------------------
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures"
for pasta in [DATA_DIR, MODEL_DIR, REPORT_DIR, FIGURE_DIR]:
    pasta.mkdir(parents=True, exist_ok=True)

# AJUSTE este caminho para o arquivo real (.csv, .parquet, .xlsx ou .json).
DATA_PATH = DATA_DIR / "financial_phrasebank_portuguese.csv"

SPLIT_PATH = DATA_DIR / "splits_nlp_seed42.parquet"
TRANSLATION_PATH = DATA_DIR / "translations.parquet"
RESULTS_PATH = REPORT_DIR / "comparison_models.csv"

TRANSLATION_MODEL = "Helsinki-NLP/opus-mt-tc-big-en-pt"
BERTIMBAU_MODEL = "neuralmind/bert-base-portuguese-cased"
LABELS = ["negative", "neutral", "positive"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

TEST_SIZE = 0.15
VALID_SIZE = 0.15
MAX_LENGTH_TRANSLATION = 256
MAX_LENGTH_BERT = 192
MAX_VOCAB = 20_000

MODO_RAPIDO = False
EXECUTAR_TRADUCAO = False
EXECUTAR_TFIDF = True
EXECUTAR_OPTUNA = False
EXECUTAR_BILSTM = False
EXECUTAR_EMBEDDINGS = False
EXECUTAR_LIGHTGBM = False
EXECUTAR_LORA = False

print("DATA_PATH:", DATA_PATH)
print("Modo rápido:", MODO_RAPIDO)

## 10. Funções auxiliares compartilhadas

Uma avaliação justa exige a mesma implementação de métricas para todos. A função `evaluate_predictions` recebe rótulos, probabilidades e metadados de custo, produz uma linha consolidada e exibe matriz de confusão.

Observe que:

- `log_loss` exige probabilidades para todas as classes;
- o tempo de inferência é medido no conjunto inteiro;
- parâmetros treináveis contam apenas tensores com `requires_grad=True`;
- métricas do teste só são calculadas depois que a configuração está congelada.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. `clean_text_classic` executa apenas normalização justificável para modelos clássicos.
# 2. `count_trainable_parameters` ignora pesos congelados e mede a capacidade realmente ajustada.
# 3. `evaluate_predictions` recebe probabilidades; dela deriva classes via `argmax`.
# 4. Todas as médias são Macro e `zero_division=0` evita exceções em classes nunca previstas.
# 5. Log Loss avalia a confiança, enquanto F1 avalia decisões discretas.
# 6. Tempos e parâmetros são guardados na mesma linha das métricas.
# 7. `all_results` e `all_test_predictions` permitem comparação posterior sem recalcular modelos.
# Saída esperada: funções comuns que garantem protocolo idêntico.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Definir limpeza moderada, contagem de parâmetros e avaliação comum.
# -----------------------------------------------------------------------------
def clean_text_classic(text):
    """Limpeza moderada para TF-IDF/BiLSTM; preserva sinais úteis."""
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def evaluate_predictions(
    y_true, proba, model_name, input_track,
    train_seconds=np.nan, inference_seconds=np.nan,
    translation_seconds=0.0, trainable_params=np.nan,
    show_matrix=True,
):
    proba = np.asarray(proba)
    pred = proba.argmax(axis=1)
    y_true = np.asarray(y_true)

    row = {
        "model": model_name,
        "input_track": input_track,
        "accuracy": accuracy_score(y_true, pred),
        "precision_macro": precision_score(y_true, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, pred, average="macro", zero_division=0),
        "log_loss": log_loss(y_true, proba, labels=list(range(len(LABELS)))),
        "train_seconds": train_seconds,
        "inference_seconds": inference_seconds,
        "translation_seconds": translation_seconds,
        "pipeline_seconds": inference_seconds + translation_seconds,
        "trainable_params": trainable_params,
    }
    f1_classes = f1_score(
        y_true, pred, average=None, labels=list(range(len(LABELS))), zero_division=0
    )
    row.update({f"f1_{label}": score for label, score in zip(LABELS, f1_classes)})

    if show_matrix:
        cm = confusion_matrix(y_true, pred, labels=list(range(len(LABELS))))
        plt.figure(figsize=(5.5, 4.2))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=LABELS, yticklabels=LABELS)
        plt.xlabel("Previsto")
        plt.ylabel("Verdadeiro")
        plt.title(f"{model_name} — {input_track}")
        plt.tight_layout()
        plt.show()

    return row, pred


def save_results(rows):
    results = pd.DataFrame(rows).sort_values("f1_macro", ascending=False)
    results.to_csv(RESULTS_PATH, index=False)
    return results


all_results = []
all_test_predictions = {}

## 11. Leitura, normalização do esquema e validação

O dataset deve conter:

- uma frase original em inglês;
- `text_pt`, tradução portuguesa de referência;
- `y`, rótulo de sentimento.

Como o nome da coluna inglesa pode variar, o código procura candidatos conhecidos e normaliza para `text_en`. Essa detecção é explícita: se nenhuma coluna for encontrada, o notebook interrompe com uma mensagem útil.

Validações antecedem qualquer split:

- textos ausentes ou vazios;
- rótulos fora do contrato;
- duplicatas;
- traduções idênticas ao inglês;
- identificador estável por linha.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. `read_table` escolhe o leitor pelo sufixo, sem presumir CSV.
# 2. `normalize_schema` procura nomes plausíveis para o inglês e normaliza para `text_en`.
# 3. O recorte de colunas impede que variáveis não autorizadas entrem acidentalmente no modelo.
# 4. Textos e rótulos são convertidos para strings e têm espaços externos removidos.
# 5. IDs numéricos 0/1/2 são aceitos somente por um mapeamento explícito.
# 6. Textos vazios e rótulos desconhecidos interrompem o fluxo em vez de serem ocultados.
# 7. `example_id` nasce antes do split e será a chave de alinhamento dos artefatos.
# Saída esperada: DataFrame canônico com `example_id`, textos e rótulo.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Ler formatos comuns, localizar a coluna inglesa e validar o contrato dos dados.
# -----------------------------------------------------------------------------
def read_table(path):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".json":
        return pd.read_json(path)
    raise ValueError(f"Formato não suportado: {suffix}")


def normalize_schema(raw):
    df = raw.copy()
    candidates_en = [
        "text_en", "text", "sentence", "phrase", "english",
        "sentence_en", "text_english", "original"
    ]
    en_col = next((c for c in candidates_en if c in df.columns), None)
    if en_col is None:
        raise KeyError(
            "Coluna inglesa não encontrada. Colunas disponíveis: "
            + ", ".join(map(str, df.columns))
        )
    if en_col != "text_en":
        df = df.rename(columns={en_col: "text_en"})

    required = {"text_en", "text_pt", "y"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Colunas obrigatórias ausentes: {sorted(missing)}")

    df = df[["text_en", "text_pt", "y"]].copy()
    for col in ["text_en", "text_pt", "y"]:
        df[col] = df[col].astype(str).str.strip()

    # Aceita rótulos textuais ou IDs 0/1/2, mas normaliza para strings canônicas.
    numeric_map = {"0": "negative", "1": "neutral", "2": "positive"}
    df["y"] = df["y"].str.lower().replace(numeric_map)

    if df[["text_en", "text_pt"]].eq("").any().any():
        raise ValueError("Há textos vazios; trate-os explicitamente antes de prosseguir.")
    invalid = sorted(set(df["y"]) - set(LABELS))
    if invalid:
        raise ValueError(f"Rótulos fora do contrato: {invalid}")

    df.insert(0, "example_id", np.arange(len(df), dtype=int))
    return df


if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {DATA_PATH}\n"
        "Ajuste DATA_PATH na configuração e execute novamente."
    )

df = normalize_schema(read_table(DATA_PATH))
print("Shape:", df.shape)
display(df.head())
display(df["y"].value_counts().rename_axis("classe").to_frame("quantidade"))

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. O primeiro bloco mede duplicação em diferentes níveis para tornar a decisão auditável.
# 2. Agrupar `text_en` e contar rótulos distintos identifica contradições supervisionadas.
# 3. Contradições não são resolvidas automaticamente porque exigem julgamento sobre os dados.
# 4. Duplicatas totalmente idênticas são removidas para não atravessarem splits.
# 5. Após a remoção, recriamos IDs contíguos; nenhum artefato havia sido persistido ainda.
# 6. A tradução não ocorre antes desta auditoria, evitando gastar computação com linhas descartadas.
# Saída esperada: corpus deduplicado ou erro explícito em caso de conflito.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Auditar duplicatas e possíveis inconsistências antes de criar os splits.
# -----------------------------------------------------------------------------
audit = pd.Series({
    "linhas": len(df),
    "textos_en_duplicados": df.duplicated("text_en").sum(),
    "textos_pt_duplicados": df.duplicated("text_pt").sum(),
    "pares_en_pt_duplicados": df.duplicated(["text_en", "text_pt"]).sum(),
    "pares_com_rotulo_duplicados": df.duplicated(["text_en", "text_pt", "y"]).sum(),
    "en_igual_pt": (df["text_en"].str.casefold() == df["text_pt"].str.casefold()).sum(),
})
display(audit.to_frame("valor"))

# Duplicatas exatas com o mesmo rótulo não acrescentam informação e podem cruzar
# os splits. Mantemos a primeira ocorrência. Conflitos de rótulo são bloqueados.
conflicts = df.groupby("text_en")["y"].nunique()
conflicting_texts = conflicts[conflicts > 1]
if len(conflicting_texts):
    raise ValueError(
        f"{len(conflicting_texts)} textos ingleses têm rótulos conflitantes. "
        "Resolva-os antes do benchmark."
    )

df = df.drop_duplicates(["text_en", "text_pt", "y"]).reset_index(drop=True)
df["example_id"] = np.arange(len(df), dtype=int)
print("Linhas após deduplicação exata:", len(df))

## 12. Split único, estratificado e persistente

Criamos o teste primeiro. Depois, dividimos o restante em treino e validação. Se $p_{test}=0{,}15$ e $p_{valid}=0{,}15$, a fração relativa de validação dentro do restante é:

$$
p'_{valid}=\frac{0{,}15}{1-0{,}15}.
$$

`stratify=y` busca preservar a proporção das classes. O arquivo persistido contém o identificador e a coluna `split`, tornando o protocolo verificável.

> O texto de teste pode ser traduzido por um tradutor congelado, pois tradução é uma transformação não supervisionada sem acesso a `y`. Contudo, seu rótulo continua proibido para seleção.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. Se o split já existe, ele é recarregado; o notebook não sorteia novamente.
# 2. Comparar conjuntos de IDs detecta alteração do dataset antes de mesclar.
# 3. O teste é separado primeiro com estratificação por rótulo.
# 4. A validação relativa corrige o denominador depois da retirada do teste.
# 5. Somente `example_id` e `split` são persistidos, evitando duplicar textos no artefato.
# 6. As asserções verificam ausência de splits vazios, IDs repetidos ou linhas sem partição.
# Saída esperada: DataFrame com uma e somente uma partição para cada exemplo.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Criar uma única partição ou recarregar exatamente a partição já persistida.
# -----------------------------------------------------------------------------
def create_or_load_splits(df, split_path):
    if split_path.exists():
        saved = pd.read_parquet(split_path)
        expected = set(df["example_id"])
        observed = set(saved["example_id"])
        if expected != observed:
            raise ValueError(
                "O dataset atual não corresponde ao split salvo. "
                "Não sobrescreva silenciosamente: investigue a mudança."
            )
        return df.merge(saved[["example_id", "split"]], on="example_id", how="left")

    train_valid, test = train_test_split(
        df, test_size=TEST_SIZE, random_state=SEED, stratify=df["y"]
    )
    valid_relative = VALID_SIZE / (1 - TEST_SIZE)
    train, valid = train_test_split(
        train_valid, test_size=valid_relative,
        random_state=SEED, stratify=train_valid["y"]
    )

    split_map = pd.concat([
        train[["example_id"]].assign(split="train"),
        valid[["example_id"]].assign(split="valid"),
        test[["example_id"]].assign(split="test"),
    ], ignore_index=True).sort_values("example_id")
    split_map.to_parquet(split_path, index=False)
    return df.merge(split_map, on="example_id", how="left")


df = create_or_load_splits(df, SPLIT_PATH)
split_distribution = pd.crosstab(df["split"], df["y"], margins=True)
display(split_distribution)

assert df["split"].notna().all()
assert set(df["split"]) == {"train", "valid", "test"}
assert not df["example_id"].duplicated().any()

## 13. Tradução automática do inglês para o português

O tradutor é um componente congelado:

$$
\hat{x}_{pt}=g_{\phi}(x_{en}),
\qquad \phi\ \text{fixo}.
$$

Ele não recebe `y`, não é ajustado no corpus e usa a mesma configuração para todos os exemplos. O cache contém `example_id`, textos, rótulo e tempo aproximado, evitando traduções diferentes entre modelos.

### Por que comparar com `text_pt`?

`text_pt` funciona como uma aproximação de referência. A diferença

$$
\Delta F1 = F1(\texttt{text\_pt\_mt})-F1(\texttt{text\_pt})
$$

mede a sensibilidade do classificador ao texto gerado. Ela não é uma métrica pura de tradução, mas é diretamente ligada à tarefa final.

### Como funciona o tradutor encoder–decoder?

O tradutor é um modelo seq2seq. O **encoder** recebe a frase inglesa e produz estados contextuais:

$$
H^{enc}=\operatorname{Encoder}(x_{en}).
$$

O **decoder** produz a tradução autoregressivamente. No passo $t$:

$$
p(y_t\mid y_{<t},x_{en})
=\operatorname{softmax}(W h_t^{dec}+b).
$$

Cada novo token depende da fonte e dos tokens portugueses já produzidos. A probabilidade de uma tradução completa é fatorada como:

$$
p(y_{1:T}\mid x)
=\prod_{t=1}^{T}p(y_t\mid y_{<t},x).
$$

Usamos `do_sample=False` para geração determinística. `num_beams=4` mantém quatro hipóteses parciais e busca uma sequência de maior probabilidade aproximada. Beam search não garante a melhor tradução linguística e pode preferir sequências genéricas, mas oferece um compromisso simples e reproduzível.

```text
Batch de frases inglesas
        ↓ tokenizer do tradutor
input_ids + attention_mask
        ↓ encoder
representações contextuais da fonte
        ↓ decoder autoregressivo + beam search
IDs em português
        ↓ batch_decode
text_pt_mt
```

#### O que pode dar errado?

- uma negação pode ser omitida;
- uma relação temporal pode mudar;
- abreviações financeiras podem ser traduzidas literalmente;
- números podem ser reformatados;
- `margin`, `share`, `equity` e `yield` dependem do contexto;
- sequências longas podem ser truncadas.

O cache preserva o resultado e registra o checkpoint usado. Idealmente, um ambiente de produção também registra revisão/commit do modelo, versão do Transformers, parâmetros de geração e hash do texto-fonte.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. Tokenizer e tradutor são carregados do mesmo checkpoint seq2seq.
# 2. `model.eval()` e `torch.inference_mode()` desativam dropout e gradientes.
# 3. O laço usa batches para aproveitar paralelismo e limitar memória.
# 4. Padding ocorre dentro de cada batch; truncamento respeita o limite configurado.
# 5. Beam search usa quatro hipóteses e `do_sample=False`, tornando a geração determinística.
# 6. `batch_decode` remove tokens especiais e devolve strings portuguesas.
# 7. O rótulo nunca é passado ao modelo; ele aparece no cache apenas para alinhamento posterior.
# 8. Se o cache existe, ele é reutilizado, garantindo que todos os modelos vejam a mesma tradução.
# Saída esperada: `translations.parquet` com `text_pt_mt` e metadados.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Gerar ou carregar traduções em cache, sem utilizar o rótulo no tradutor.
# -----------------------------------------------------------------------------
def generate_translations(frame, output_path, model_name, batch_size=16):
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE)
    model.eval()

    texts = frame["text_en"].tolist()
    translated = []
    start = time.perf_counter()

    for begin in range(0, len(texts), batch_size):
        batch = texts[begin:begin + batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True, truncation=True,
            max_length=MAX_LENGTH_TRANSLATION
        ).to(DEVICE)
        with torch.inference_mode():
            generated = model.generate(
                **inputs, max_new_tokens=MAX_LENGTH_TRANSLATION,
                num_beams=4, do_sample=False
            )
        translated.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))

    elapsed = time.perf_counter() - start
    output = frame[["example_id", "text_en", "text_pt", "y", "split"]].copy()
    output["text_pt_mt"] = translated
    output["translation_model"] = model_name
    output["translation_total_seconds"] = elapsed
    output.to_parquet(output_path, index=False)
    print(f"{len(output)} frases traduzidas em {elapsed:.1f}s")
    return output


if TRANSLATION_PATH.exists():
    translated_df = pd.read_parquet(TRANSLATION_PATH)
elif EXECUTAR_TRADUCAO:
    translated_df = generate_translations(
        df, TRANSLATION_PATH, TRANSLATION_MODEL,
        batch_size=4 if MODO_RAPIDO else 16
    )
else:
    raise FileNotFoundError(
        f"Cache ausente: {TRANSLATION_PATH}. "
        "Ative EXECUTAR_TRADUCAO=True para gerá-lo."
    )

required_translation_cols = {
    "example_id", "text_en", "text_pt", "text_pt_mt", "y", "split"
}
assert required_translation_cols.issubset(translated_df.columns)
assert translated_df["example_id"].is_unique
assert translated_df["text_pt_mt"].str.strip().ne("").all()
df = translated_df.copy()
display(df[["text_en", "text_pt", "text_pt_mt"]].head(5))

## 14. Validação da tradução e análise exploratória

Métricas automáticas de similaridade podem ser úteis, mas uma tradução lexicalmente diferente pode estar semanticamente correta. Por isso, fazemos verificações diretamente relacionadas ao domínio:

- números e percentuais preservados;
- negações presentes;
- comprimento relativo;
- exemplos com maior divergência;
- distribuição das classes por split.

Esses diagnósticos não escolhem o classificador usando o teste. A inspeção qualitativa detalhada para desenvolvimento deve priorizar treino e validação.

### Pré-processamento por família de modelo

Não existe um único pré-processamento ideal para todas as representações.

#### TF-IDF

Usamos `clean_text_classic`:

```text
texto → strip → minúsculas → compactação de espaços
```

Mantemos pontuação relevante, dígitos, percentuais e negações. O `TfidfVectorizer` realiza sua própria separação em termos. Minúsculas reduzem variantes artificiais como `Lucro` e `lucro`.

#### BiLSTM

Usamos uma regex explícita para obter palavras, números e pontuação. O vocabulário é criado somente no treino. Padding ocorre no `collate_fn`, isto é, dinamicamente em cada batch, reduzindo computação desperdiçada.

#### BERTimbau

O texto não passa por `clean_text_classic`. O tokenizer oficial decide:

- subpalavras;
- IDs especiais;
- truncamento;
- padding;
- máscara.

Modificar acentuação ou caixa sem necessidade produziria uma distribuição diferente daquela vista no pré-treinamento do modelo *cased*.

#### Tradução

O tradutor recebe o inglês original, não uma versão simplificada. Limpeza anterior poderia remover sinais necessários para produzir uma tradução fiel.

| Operação | TF-IDF | BiLSTM | BERTimbau | Tradutor |
|---|---:|---:|---:|---:|
| compactar espaços | sim | sim | não necessário | não |
| converter para minúsculas | sim | sim | não | não |
| preservar números/% | sim | sim | sim | sim |
| remover stopwords | não | não | não | não |
| tokenizer oficial | não | não | sim | sim |
| vocabulário ajustado no treino | sim | sim | não | não |

Essa separação evita aplicar hábitos de modelos clássicos a Transformers sem justificativa.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. Regex extrai números incluindo sinal, separador decimal e percentual.
# 2. A função de negação procura formas portuguesas com limites de palavra.
# 3. `assign` cria diagnósticos sem sobrescrever as colunas originais.
# 4. Igualdade das listas exige preservar valor e ordem dos números.
# 5. A razão de comprimento usa `clip(lower=1)` para impedir divisão por zero.
# 6. Os três gráficos comparam classes, comprimentos absolutos e compressão/expansão.
# 7. Esses sinais são heurísticas; não substituem avaliação humana ou métrica semântica.
# Saída esperada: auditoria tabular e três visualizações da tradução.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Comparar propriedades observáveis entre referência e tradução automática.
# -----------------------------------------------------------------------------
def extract_numbers(text):
    return re.findall(r"[-+]?\d+(?:[.,]\d+)?%?", str(text))

def has_negation_pt(text):
    return bool(re.search(r"\b(não|nunca|nem|sem|jamais)\b", str(text).lower()))

translation_audit = df.assign(
    len_ref=df["text_pt"].str.split().str.len(),
    len_mt=df["text_pt_mt"].str.split().str.len(),
    numbers_ref=df["text_pt"].map(extract_numbers),
    numbers_mt=df["text_pt_mt"].map(extract_numbers),
    neg_ref=df["text_pt"].map(has_negation_pt),
    neg_mt=df["text_pt_mt"].map(has_negation_pt),
)
translation_audit["numbers_preserved"] = (
    translation_audit["numbers_ref"] == translation_audit["numbers_mt"]
)
translation_audit["negation_agrees"] = (
    translation_audit["neg_ref"] == translation_audit["neg_mt"]
)
translation_audit["length_ratio"] = (
    translation_audit["len_mt"] / translation_audit["len_ref"].clip(lower=1)
)

display(translation_audit[[
    "numbers_preserved", "negation_agrees", "length_ratio"
]].describe(include="all"))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=df, x="y", hue="split", order=LABELS, ax=axes[0])
axes[0].set_title("Classes por split")
sns.histplot(data=translation_audit, x="len_ref", bins=30, ax=axes[1], label="referência")
sns.histplot(data=translation_audit, x="len_mt", bins=30, ax=axes[1], label="automática", alpha=.55)
axes[1].legend()
axes[1].set_title("Comprimento em palavras")
sns.histplot(data=translation_audit, x="length_ratio", bins=30, ax=axes[2])
axes[2].axvline(1, color="black", ls="--")
axes[2].set_title("Razão comprimento MT / referência")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. A máscara seleciona automaticamente casos com sinais objetivos de fragilidade.
# 2. Operador `|` significa que basta um critério: número, negação ou comprimento extremo.
# 3. Mantemos inglês, referência e tradução lado a lado para inspeção causal.
# 4. `.head(20)` limita somente a exibição; o DataFrame `fragile` conserva todos os casos.
# 5. A seleção não consulta acerto de teste e não influencia hiperparâmetros.
# Saída esperada: amostra reproduzível de traduções que merecem revisão.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Exibir casos potencialmente frágeis da tradução, sem seleção por desempenho.
# -----------------------------------------------------------------------------
fragile = translation_audit[
    (~translation_audit["numbers_preserved"])
    | (~translation_audit["negation_agrees"])
    | (translation_audit["length_ratio"] < 0.55)
    | (translation_audit["length_ratio"] > 1.8)
].copy()

display(fragile[[
    "example_id", "split", "text_en", "text_pt", "text_pt_mt",
    "numbers_ref", "numbers_mt", "neg_ref", "neg_mt", "length_ratio"
]].head(20))
print("Casos sinalizados:", len(fragile), "de", len(df))

### Checkpoint conceitual 2

- O tradutor é uma etapa fixa e independente dos rótulos.
- Cachear evita que aleatoriedade ou mudança de versão altere a entrada.
- O split pertence ao exemplo, não à representação.
- Preservar números não garante preservar sentido, mas detecta erros relevantes.
- A comparação referência × automática mede robustez ponta a ponta.
- Nenhuma inspeção do teste deve orientar hiperparâmetros.

# Parte II — Modelagem e benchmark

## 15. Protocolo experimental comum

Para cada modelo e para cada trilha de texto:

1. ajuste usando somente `train`;
2. escolha configuração e early stopping por `valid`;
3. congele a decisão;
4. avalie uma única vez em `test`;
5. registre métricas, tempos e parâmetros.

As trilhas são:

| Nome | Entrada | Papel |
|---|---|---|
| `reference_pt` | `text_pt` | controle |
| `machine_translated_pt` | `text_pt_mt` | pipeline principal |

O teste não será concatenado ao treino depois da seleção, porque isso apagaria a separação necessária para a comparação honesta.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. Filtramos o DataFrame pela coluna persistida, não por novo sorteio.
# 2. `reset_index` cria índices locais convenientes, mas `example_id` continua sendo a chave global.
# 3. O rótulo textual vira inteiro somente após o split; a ordem vem de `LABEL2ID`.
# 4. O tempo de tradução do teste é estimado proporcionalmente e marcado como aproximação.
# 5. A baseline prevê a classe majoritária aprendida exclusivamente no treino.
# 6. Probabilidades one-hot permitem usar a mesma função de avaliação dos demais modelos.
# Saída esperada: três subconjuntos e um piso de desempenho sem representação textual.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Materializar os três subconjuntos e uma baseline de classe majoritária.
# -----------------------------------------------------------------------------
train_df = df[df["split"] == "train"].reset_index(drop=True)
valid_df = df[df["split"] == "valid"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

for part in [train_df, valid_df, test_df]:
    part["label_id"] = part["y"].map(LABEL2ID).astype(int)

# O cache registra o tempo total. Para o teste, usamos uma alocação proporcional
# ao número de exemplos; isso é uma aproximação explícita, não uma nova medição.
if "translation_total_seconds" in df:
    TRANSLATION_TEST_SECONDS = (
        float(df["translation_total_seconds"].iloc[0])
        * len(test_df) / len(df)
    )
else:
    TRANSLATION_TEST_SECONDS = np.nan

majority_id = train_df["label_id"].mode().iloc[0]
majority_proba = np.zeros((len(test_df), len(LABELS)), dtype=float)
majority_proba[:, majority_id] = 1.0
baseline_row, _ = evaluate_predictions(
    test_df["label_id"], majority_proba,
    model_name="Classe majoritária", input_track="sem texto",
    trainable_params=0
)
display(pd.DataFrame([baseline_row]))

## 16. TF-IDF + Regressão Logística

**Pergunta de aprendizagem:** quão longe chegamos com uma representação esparsa e um modelo linear?

Pipeline:

```text
texto → limpeza moderada → TF-IDF de unigramas/bigramas
      → regressão logística → probabilidades
```

O `Pipeline` impede vazamento porque o TF-IDF é ajustado dentro do treino. Faremos poucos ajustes moderados em `C`, `max_features` e `class_weight`, escolhidos pelo F1 Macro de validação.

Os coeficientes de uma classe indicam quais termos aumentam seus logits, mantidas as demais variáveis.

### Fluxo completo, dimensões e seleção

Para $N_{train}$ documentos e vocabulário final de tamanho $V$:

```text
N textos
    ↓ clean_text_classic
N strings normalizadas
    ↓ TfidfVectorizer.fit_transform (somente treino)
X_train: (N_train, V), matriz esparsa
    ↓ LogisticRegression.fit
W: (3, V) e b: (3,)
    ↓ predict_proba
P: (N, 3)
```

O `Pipeline` chama `fit` em cada etapa durante o treino. Em validação e teste, chama apenas `transform` no TF-IDF e `predict_proba` no classificador. Essa diferença é essencial:

$$
\underbrace{\operatorname{fit}_{train}}_{\text{aprende vocabulário/IDF/pesos}}
\neq
\underbrace{\operatorname{transform}_{valid,test}}_{\text{aplica estado congelado}}.
$$

Testamos poucas configurações. Para cada uma:

```text
ajusta em train
→ prevê valid
→ calcula F1 Macro de valid
→ guarda modelo e escore
```

Depois escolhemos o maior F1 de validação e consultamos o teste uma vez. Não juntamos treino e validação antes do teste porque isso alteraria o modelo já selecionado e exigiria uma nova regra explícita para determinar épocas/hiperparâmetros.

`class_weight="balanced"` repondera aproximadamente cada classe por:

$$
w_k=\frac{N}{K\,N_k}.
$$

Isso pode ajudar classes raras, mas também piorar calibração. Por isso é candidato, não regra automática.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. A função recebe a coluna de texto, permitindo repetir exatamente o fluxo nas duas trilhas.
# 2. Cada candidato cria um Pipeline novo; vocabulário, IDF e pesos não são compartilhados.
# 3. `min_df`, bigramas e TF sublinear permanecem fixos para limitar a pergunta experimental.
# 4. Cada modelo ajusta somente `train` e é ranqueado somente por F1 Macro de `valid`.
# 5. `np.argmax` escolhe o índice do candidato, não uma configuração derivada do teste.
# 6. Cronômetros cercam separadamente `fit` e `predict_proba`.
# 7. O teste é usado uma vez pelo modelo já escolhido e suas previsões são registradas.
# 8. O número de parâmetros inclui coeficientes e interceptos da regressão.
# Saída esperada: modelo escolhido, tabela de tentativas, métricas e probabilidades.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Selecionar uma configuração moderada no conjunto de validação e avaliar no teste.
# -----------------------------------------------------------------------------
def run_tfidf_track(text_col, track_name):
    candidates = [
        {"C": 1.0, "max_features": 20_000, "class_weight": None},
        {"C": 2.0, "max_features": 30_000, "class_weight": None},
        {"C": 1.0, "max_features": 30_000, "class_weight": "balanced"},
    ]
    if MODO_RAPIDO:
        candidates = candidates[:1]

    trials = []
    fitted = []
    for params in candidates:
        pipe = Pipeline([
            ("tfidf", TfidfVectorizer(
                preprocessor=clean_text_classic,
                ngram_range=(1, 2), min_df=2,
                max_features=params["max_features"],
                sublinear_tf=True
            )),
            ("clf", LogisticRegression(
                C=params["C"], class_weight=params["class_weight"],
                max_iter=2_000, solver="lbfgs", random_state=SEED
            )),
        ])
        start = time.perf_counter()
        pipe.fit(train_df[text_col], train_df["label_id"])
        train_seconds = time.perf_counter() - start
        valid_proba = pipe.predict_proba(valid_df[text_col])
        valid_f1 = f1_score(
            valid_df["label_id"], valid_proba.argmax(1), average="macro"
        )
        trials.append({**params, "valid_f1_macro": valid_f1})
        fitted.append((pipe, train_seconds))

    best_idx = int(np.argmax([x["valid_f1_macro"] for x in trials]))
    best_model, train_seconds = fitted[best_idx]

    start = time.perf_counter()
    test_proba = best_model.predict_proba(test_df[text_col])
    infer_seconds = time.perf_counter() - start
    row, pred = evaluate_predictions(
        test_df["label_id"], test_proba,
        "TF-IDF + Regressão Logística", track_name,
        train_seconds=train_seconds, inference_seconds=infer_seconds,
        translation_seconds=(
            TRANSLATION_TEST_SECONDS
            if track_name == "machine_translated_pt" else 0.0
        ),
        trainable_params=(
            best_model.named_steps["clf"].coef_.size
            + best_model.named_steps["clf"].intercept_.size
        )
    )
    return best_model, pd.DataFrame(trials), row, pred, test_proba


tfidf_artifacts = {}
if EXECUTAR_TFIDF:
    for text_col, track in [
        ("text_pt", "reference_pt"),
        ("text_pt_mt", "machine_translated_pt"),
    ]:
        model, trials, row, pred, proba = run_tfidf_track(text_col, track)
        tfidf_artifacts[track] = model
        all_results.append(row)
        all_test_predictions[("TF-IDF + Regressão Logística", track)] = pred
        print("\nTrilha:", track)
        display(trials.sort_values("valid_f1_macro", ascending=False))

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. Recuperamos nomes de termos na mesma ordem das colunas do TF-IDF.
# 2. Cada linha de `coef_` corresponde a uma das três classes.
# 3. `argsort` ordena índices; os quinze últimos têm os maiores pesos positivos.
# 4. A posição é calculada dentro de cada classe antes do pivot.
# 5. O resultado mostra ranking e magnitude lado a lado.
# 6. Coeficientes descrevem associação do modelo, não importância causal no mercado.
# Saída esperada: tabela interpretável dos termos que mais elevam cada logit.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Interpretar os termos com maior coeficiente em cada classe.
# -----------------------------------------------------------------------------
if EXECUTAR_TFIDF:
    chosen = tfidf_artifacts["machine_translated_pt"]
    vectorizer = chosen.named_steps["tfidf"]
    classifier = chosen.named_steps["clf"]
    terms = np.asarray(vectorizer.get_feature_names_out())

    rows = []
    for class_idx, label in enumerate(LABELS):
        top_idx = np.argsort(classifier.coef_[class_idx])[-15:][::-1]
        rows.extend({
            "classe": label,
            "termo": terms[i],
            "coeficiente": classifier.coef_[class_idx, i],
        } for i in top_idx)
    coefficients = pd.DataFrame(rows)
    coefficients["posição"] = coefficients.groupby("classe").cumcount() + 1
    display(coefficients.pivot(
        index="posição", columns="classe", values=["termo", "coeficiente"]
    ))

## 17. BiLSTM treinada do zero

**Pergunta de aprendizagem:** o corpus possui dados suficientes para aprender embeddings e dependências sequenciais sem pré-treinamento?

O vocabulário é construído apenas em `train`. A sequência passa por:

$$
\text{IDs}\rightarrow
\text{Embedding}\rightarrow
\text{BiLSTM}\rightarrow
[h_{\rightarrow};h_{\leftarrow}]
\rightarrow\text{Dropout}\rightarrow\text{Linear}.
$$

Treinamento:

- `CrossEntropyLoss`;
- Adam;
- weight decay;
- clipping da norma do gradiente;
- early stopping pelo F1 Macro de validação.

Padding é ignorado via `padding_idx`; os comprimentos permitem empacotar sequências reais com `pack_padded_sequence`.

### Fluxo completo do batch na BiLSTM

Considere batch $N=64$, sequência máxima local $L=40$, embedding $d_e=128$ e estado oculto por direção $d_h=128$:

```text
token_ids
(64, 40)
    ↓ nn.Embedding
(64, 40, 128)
    ↓ pack_padded_sequence
somente posições reais são processadas
    ↓ BiLSTM
hidden: (2 × num_layers, 64, 128)
    ↓ últimos estados forward/backward
(64, 128) + (64, 128)
    ↓ concatenação
(64, 256)
    ↓ Dropout
(64, 256)
    ↓ Linear(256, 3)
(64, 3 logits)
```

Não há uma BiLSTM por frase. Uma única rede, com pesos compartilhados, processa todas as sequências do batch. O eixo do batch nunca é concatenado com as características.

#### Por que empacotar?

Sem packing, a LSTM processaria vários `<PAD>` depois do fim real. Com:

```python
pack_padded_sequence(..., enforce_sorted=False)
```

o PyTorch ignora posições artificiais, mantendo a ordem original do batch.

#### Regularização e estabilidade

- **Dropout:** zera ativações aleatoriamente no treino;
- **weight decay:** penaliza pesos grandes;
- **gradient clipping:** limita $\lVert g\rVert_2$ para evitar explosão;
- **early stopping:** restaura o estado com melhor F1 de validação.

O clipping utilizado é:

$$
g\leftarrow g\cdot
\min\left(1,\frac{\tau}{\lVert g\rVert_2}\right),
\qquad \tau=1.
$$

#### Uma atualização por batch

```text
forward → logits → CrossEntropy média
→ zero_grad → backward → clipping
→ Adam.step → uma atualização dos pesos
```

O estado oculto final resume a sequência, mas essa compressão pode perder detalhes de frases longas. Mantemos esse pooling fixo para que o Optuna não transforme o projeto em comparação de arquiteturas.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. A regex preserva palavras Unicode, números decimais/percentuais e pontuação.
# 2. `Counter` é alimentado somente pelos textos de treino passados à função.
# 3. `<PAD>=0` coincide com `padding_idx`; `<UNK>=1` recebe palavras fora do vocabulário.
# 4. `min_freq` e `max_vocab` controlam ruído e memória.
# 5. O Dataset converte uma frase sob demanda, evitando guardar todos os tensores duplicados.
# 6. Sequência vazia recebe `<UNK>` para que a LSTM sempre veja ao menos um passo.
# 7. `collate_sequences` calcula comprimentos antes do padding e devolve batch-first.
# Saída esperada: batches `(ids, lengths, labels)` adequados à BiLSTM.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Construir vocabulário, Dataset e função de collation para a BiLSTM.
# -----------------------------------------------------------------------------
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset

PAD_TOKEN, UNK_TOKEN = "<PAD>", "<UNK>"

def word_tokenize_pt(text):
    return re.findall(r"\w+(?:[.,]\d+)?%?|[^\w\s]", clean_text_classic(text),
                      flags=re.UNICODE)

def build_vocab(texts, max_vocab=MAX_VOCAB, min_freq=2):
    counts = Counter(token for text in texts for token in word_tokenize_pt(text))
    vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    for token, freq in counts.most_common():
        if freq < min_freq or len(vocab) >= max_vocab:
            break
        vocab[token] = len(vocab)
    return vocab

class TextSequenceDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = list(texts)
        self.labels = list(labels)
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = [self.vocab.get(t, self.vocab[UNK_TOKEN])
               for t in word_tokenize_pt(self.texts[idx])]
        ids = ids or [self.vocab[UNK_TOKEN]]
        return torch.tensor(ids, dtype=torch.long), int(self.labels[idx])

def collate_sequences(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(x) for x in sequences], dtype=torch.long)
    padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    return padded, lengths, torch.tensor(labels, dtype=torch.long)

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. Embedding transforma IDs `(N,L)` em vetores `(N,L,d_e)`.
# 2. `padding_idx=0` impede atualização útil do vetor reservado ao padding.
# 3. `bidirectional=True` cria uma recorrência forward e outra backward por camada.
# 4. Dropout interno da LSTM só é válido quando há mais de uma camada.
# 5. Packing remove passos artificiais sem ordenar manualmente o batch.
# 6. `hidden[-2]` e `hidden[-1]` são as duas direções da última camada.
# 7. Concatenar produz `2*hidden_dim`; a Linear converte esse vetor em três logits.
# Saída esperada: tensor `(batch_size, 3)` sem softmax.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Definir explicitamente a arquitetura BiLSTM e sua passagem direta.
# -----------------------------------------------------------------------------
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim,
                 num_layers, dropout, num_classes=3):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=0
        )
        self.lstm = nn.LSTM(
            embedding_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, token_ids, lengths):
        embedded = self.embedding(token_ids)
        packed = pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (hidden, _) = self.lstm(packed)
        # Última camada: penúltimo estado = forward; último = backward.
        representation = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(self.dropout(representation))

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. `predict_torch` coloca a rede em avaliação e desliga o grafo de gradientes.
# 2. Softmax é aplicado apenas para avaliação, pois o treino entrega logits à CrossEntropy.
# 3. No treino, `zero_grad(set_to_none=True)` evita acumular gradientes entre batches.
# 4. A loss é multiplicada pelo tamanho do batch para calcular média correta por exemplo.
# 5. Clipping ocorre depois do backward e antes do passo do Adam.
# 6. A validação roda após cada época e mede F1 Macro, não loss de treino.
# 7. O melhor `state_dict` é copiado para CPU; épocas piores não sobrescrevem o checkpoint.
# 8. Ao final, o melhor estado é restaurado, não necessariamente o da última época.
# Saída esperada: melhor modelo, histórico por época e melhor F1 de validação.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Implementar treino, validação, clipping, early stopping e probabilidades.
# -----------------------------------------------------------------------------
def predict_torch(model, loader):
    model.eval()
    probabilities, labels = [], []
    with torch.inference_mode():
        for ids, lengths, y in loader:
            logits = model(ids.to(DEVICE), lengths)
            probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())
            labels.append(y.numpy())
    return np.vstack(probabilities), np.concatenate(labels)


def train_bilstm_model(model, train_loader, valid_loader, lr, weight_decay,
                       max_epochs=20, patience=4):
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    criterion = nn.CrossEntropyLoss()
    best_state, best_f1, wait = None, -np.inf, 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        running_loss = 0.0
        for ids, lengths, y in train_loader:
            ids, y = ids.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(ids, lengths)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += loss.item() * len(y)

        valid_proba, valid_y = predict_torch(model, valid_loader)
        valid_f1 = f1_score(valid_y, valid_proba.argmax(1), average="macro")
        history.append({
            "epoch": epoch,
            "train_loss": running_loss / len(train_loader.dataset),
            "valid_f1_macro": valid_f1,
        })

        if valid_f1 > best_f1 + 1e-4:
            best_f1 = valid_f1
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_f1

## 18. Otimização da BiLSTM com Optuna

Optuna otimiza somente:

$$
\{\text{embedding\_dim},\text{hidden\_dim},\text{dropout},
\eta,\lambda,\text{batch\_size},\text{num\_layers}\}.
$$

A arquitetura permanece BiLSTM, o otimizador permanece Adam e o objetivo é F1 Macro de validação. O teste não aparece na função `objective`.

Cada trilha textual pode produzir hiperparâmetros diferentes, pois erros de tradução alteram vocabulário e distribuição. Para comparar robustez de forma mais rígida, também é possível selecionar na referência e reaplicar a mesma configuração à tradução; o notebook registra claramente a opção adotada.

### Cronologia de uma tentativa do Optuna

O problema de otimização é:

$$
\theta^*
=\arg\max_{\theta\in\Theta}
F1_{\mathrm{macro}}^{valid}(\theta).
$$

`θ` contém apenas os sete hiperparâmetros autorizados. Uma tentativa executa:

```text
TPE sugere θ
    ↓
redefine seed
    ↓
cria DataLoaders com batch_size de θ
    ↓
inicializa uma NOVA BiLSTM com pesos aleatórios
    ↓
treina somente em train
    ↓
early stopping acompanha valid
    ↓
retorna o melhor F1 Macro de valid
    ↓
descarta a rede da tentativa
```

Reinicializar a rede é obrigatório: reutilizar pesos entre tentativas daria vantagem indevida às configurações posteriores.

| Hiperparâmetro | Efeito |
|---|---|
| `embedding_dim` | capacidade do vetor de cada token |
| `hidden_dim` | capacidade da memória em cada direção |
| `dropout` | intensidade de regularização |
| `learning_rate` | tamanho inicial das atualizações |
| `weight_decay` | penalização L2 aproximada no Adam |
| `batch_size` | ruído do gradiente e uso de memória |
| `num_layers` | profundidade recorrente |

O sampler TPE modela separadamente regiões de bons e maus resultados observados e sugere pontos com maior razão de densidades. Ele é mais direcionado que uma busca aleatória, mas não elimina incerteza: o resultado depende do espaço, orçamento, seed e ruído de treinamento.

Após a busca, criamos outra BiLSTM com a melhor configuração, treinamos com early stopping e só então avaliamos o teste.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. `make_bilstm_loaders` constrói Datasets distintos sobre um vocabulário comum de treino.
# 2. Somente o loader de treino embaralha; validação e teste mantêm ordem estável.
# 3. Um Generator semeado torna o primeiro embaralhamento reproduzível.
# 4. Cada trial redefine todas as seeds antes de inicializar pesos.
# 5. Optuna sugere apenas os sete hiperparâmetros autorizados.
# 6. Uma rede nova é criada para cada trial e descartada depois de devolver o escore.
# 7. O objetivo retorna exclusivamente F1 de validação.
# 8. `DEFAULT_BILSTM_PARAMS` possibilita testar o pipeline sem executar a busca.
# Saída esperada: Study do Optuna e vocabulário da trilha.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Otimizar e treinar a BiLSTM final para uma trilha textual.
# -----------------------------------------------------------------------------
def make_bilstm_loaders(text_col, vocab, batch_size):
    datasets = {
        "train": TextSequenceDataset(train_df[text_col], train_df["label_id"], vocab),
        "valid": TextSequenceDataset(valid_df[text_col], valid_df["label_id"], vocab),
        "test": TextSequenceDataset(test_df[text_col], test_df["label_id"], vocab),
    }
    generator = torch.Generator().manual_seed(SEED)
    return {
        "train": DataLoader(datasets["train"], batch_size=batch_size, shuffle=True,
                            collate_fn=collate_sequences, generator=generator),
        "valid": DataLoader(datasets["valid"], batch_size=batch_size, shuffle=False,
                            collate_fn=collate_sequences),
        "test": DataLoader(datasets["test"], batch_size=batch_size, shuffle=False,
                           collate_fn=collate_sequences),
    }

def optimize_bilstm(text_col, n_trials=20):
    import optuna
    vocab = build_vocab(train_df[text_col])

    def objective(trial):
        seed_everything(SEED)
        params = {
            "embedding_dim": trial.suggest_categorical("embedding_dim", [64, 128, 256]),
            "hidden_dim": trial.suggest_categorical("hidden_dim", [64, 128, 256]),
            "dropout": trial.suggest_float("dropout", 0.15, 0.50),
            "learning_rate": trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
            "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
            "num_layers": trial.suggest_int("num_layers", 1, 2),
        }
        loaders = make_bilstm_loaders(
            text_col, vocab, params["batch_size"]
        )
        model = BiLSTMClassifier(
            len(vocab), params["embedding_dim"], params["hidden_dim"],
            params["num_layers"], params["dropout"]
        ).to(DEVICE)
        _, _, best_f1 = train_bilstm_model(
            model, loaders["train"], loaders["valid"],
            params["learning_rate"], params["weight_decay"],
            max_epochs=5 if MODO_RAPIDO else 20,
            patience=2 if MODO_RAPIDO else 4
        )
        return best_f1

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study.optimize(objective, n_trials=2 if MODO_RAPIDO else n_trials)
    return study, vocab


DEFAULT_BILSTM_PARAMS = {
    "embedding_dim": 128, "hidden_dim": 128, "dropout": 0.30,
    "learning_rate": 1e-3, "weight_decay": 1e-5,
    "batch_size": 64, "num_layers": 1,
}

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. O laço repete o protocolo para referência e tradução automática.
# 2. Se Optuna estiver ativo, os melhores parâmetros são persistidos por trilha.
# 3. Caso contrário, a configuração didática padrão é declarada explicitamente.
# 4. A rede final é reinicializada e treinada com orçamento maior que os trials.
# 5. O checkpoint restaurado pelo early stopping é salvo em `.pt`.
# 6. A inferência de teste é cronometrada depois que toda seleção terminou.
# 7. O histórico exibe perda de treino e F1 de validação em painéis separados.
# 8. Previsões entram no registro comum para futura análise de erros.
# Saída esperada: dois checkpoints BiLSTM, históricos e linhas de resultado.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Executar Optuna opcionalmente e avaliar a melhor BiLSTM somente ao final.
# -----------------------------------------------------------------------------
if EXECUTAR_BILSTM:
    for text_col, track in [
        ("text_pt", "reference_pt"),
        ("text_pt_mt", "machine_translated_pt"),
    ]:
        if EXECUTAR_OPTUNA:
            study, vocab = optimize_bilstm(text_col)
            params = study.best_params
            (MODEL_DIR / f"bilstm_params_{track}.json").write_text(
                json.dumps(params, indent=2), encoding="utf-8"
            )
        else:
            params = DEFAULT_BILSTM_PARAMS.copy()
            vocab = build_vocab(train_df[text_col])

        loaders = make_bilstm_loaders(text_col, vocab, params["batch_size"])
        model = BiLSTMClassifier(
            len(vocab), params["embedding_dim"], params["hidden_dim"],
            params["num_layers"], params["dropout"]
        ).to(DEVICE)
        start = time.perf_counter()
        model, history, _ = train_bilstm_model(
            model, loaders["train"], loaders["valid"],
            params["learning_rate"], params["weight_decay"],
            max_epochs=8 if MODO_RAPIDO else 30,
            patience=2 if MODO_RAPIDO else 5
        )
        train_seconds = time.perf_counter() - start
        torch.save(model.state_dict(), MODEL_DIR / f"bilstm_{track}.pt")

        start = time.perf_counter()
        proba, y_true = predict_torch(model, loaders["test"])
        infer_seconds = time.perf_counter() - start
        row, pred = evaluate_predictions(
            y_true, proba, "BiLSTM + Optuna", track,
            train_seconds, infer_seconds,
            translation_seconds=(
                TRANSLATION_TEST_SECONDS
                if track == "machine_translated_pt" else 0.0
            ),
            trainable_params=count_trainable_parameters(model)
        )
        all_results.append(row)
        all_test_predictions[("BiLSTM + Optuna", track)] = pred

        history.plot(x="epoch", y=["train_loss", "valid_f1_macro"],
                     subplots=True, figsize=(10, 6), title=track)
        plt.tight_layout()
        plt.show()

## 19. BERTimbau congelado: embeddings contextuais

**Pergunta de aprendizagem:** uma representação linguística pré-treinada já separa sentimentos sem adaptar o Transformer?

Para cada sequência, BERTimbau retorna:

$$
H=[h_1,\ldots,h_L]\in\mathbb{R}^{L\times768}.
$$

Usaremos **mean pooling mascarado**, fixado para todo o experimento:

$$
e=\frac{\sum_{i=1}^{L}m_i h_i}{\sum_{i=1}^{L}m_i}.
$$

Os pesos do BERTimbau ficam congelados. Os embeddings são extraídos uma vez por trilha e salvos em `.npz`, sempre com `example_id`, para impedir desalinhamento.

### Do token à representação contextual

Para batch $N$, comprimento $L$ e dimensão oculta $d=768$:

```text
textos
    ↓ tokenizer BERTimbau
input_ids, attention_mask: (N, L)
    ↓ embeddings de token + posição + segmento
X⁰: (N, L, 768)
    ↓ 12 blocos Transformer congelados
H: (N, L, 768)
    ↓ mean pooling mascarado
E: (N, 768)
```

Cada bloco contém atenção multi-head, rede feed-forward, conexões residuais e normalização. Em forma simplificada:

$$
\tilde{H}^{(\ell)}
=\operatorname{LayerNorm}\left(
H^{(\ell-1)}+\operatorname{MHA}(H^{(\ell-1)})
\right),
$$

$$
H^{(\ell)}
=\operatorname{LayerNorm}\left(
\tilde{H}^{(\ell)}+\operatorname{FFN}(\tilde{H}^{(\ell)})
\right).
$$

O embedding de `banco` pode mudar entre “banco elevou o lucro” e “sentou no banco”, pois os vetores finais dependem dos demais tokens.

#### Mean pooling mascarado

Sem máscara, pads participariam da média. A implementação expande $M\in\{0,1\}^{N\times L}$ para acompanhar as 768 dimensões:

$$
E_{n,j}
=\frac{\sum_i M_{n,i}H_{n,i,j}}
{\max(1,\sum_i M_{n,i})}.
$$

`torch.inference_mode()` desliga armazenamento de gradientes; `model.eval()` desliga dropout. `requires_grad=False` documenta que nenhum parâmetro do backbone é treinável.

O cache associa embeddings a `example_id`. Conferir a igualdade dos IDs impede o erro silencioso mais perigoso desse pipeline: combinar vetor de uma frase com rótulo de outra.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. `mean_pool` expande a máscara para as 768 dimensões do estado oculto.
# 2. Multiplicação por zero remove vetores de padding antes da soma.
# 3. `clamp` protege contra divisão por zero em uma sequência degenerada.
# 4. AutoTokenizer e AutoModel vêm do mesmo checkpoint BERTimbau.
# 5. `eval`, `requires_grad=False` e `inference_mode` tornam o backbone extrator fixo.
# 6. O processamento em batches limita memória de GPU.
# 7. Vetores são convertidos para float32 antes do cache para reduzir espaço.
# 8. IDs salvos são comparados ao DataFrame antes de qualquer classificação.
# Saída esperada: matrizes `(N,768)` alinhadas e arquivos `.npz`.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Extrair embeddings com mean pooling mascarado e armazená-los por ID.
# -----------------------------------------------------------------------------
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


def extract_bert_embeddings(texts, model_name, batch_size=32):
    from transformers import AutoModel, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE)
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad = False

    output = []
    start = time.perf_counter()
    for begin in range(0, len(texts), batch_size):
        batch = list(texts[begin:begin + batch_size])
        encoded = tokenizer(
            batch, padding=True, truncation=True,
            max_length=MAX_LENGTH_BERT, return_tensors="pt"
        ).to(DEVICE)
        with torch.inference_mode():
            hidden = model(**encoded).last_hidden_state
            pooled = mean_pool(hidden, encoded["attention_mask"])
        output.append(pooled.cpu().numpy().astype("float32"))
    return np.vstack(output), time.perf_counter() - start


embedding_cache = {}
if EXECUTAR_EMBEDDINGS:
    for text_col, track in [
        ("text_pt", "reference_pt"),
        ("text_pt_mt", "machine_translated_pt"),
    ]:
        path = DATA_DIR / f"bertimbau_embeddings_{track}.npz"
        if path.exists():
            cache = np.load(path)
            embeddings, ids = cache["embeddings"], cache["example_ids"]
        else:
            embeddings, seconds = extract_bert_embeddings(
                df[text_col].tolist(), BERTIMBAU_MODEL,
                batch_size=8 if MODO_RAPIDO else 32
            )
            ids = df["example_id"].to_numpy()
            np.savez_compressed(path, embeddings=embeddings, example_ids=ids)
            print(track, "segundos:", round(seconds, 1))
        assert np.array_equal(ids, df["example_id"].to_numpy())
        embedding_cache[track] = embeddings
        print(track, embeddings.shape)

## 20. LightGBM sobre embeddings

LightGBM recebe vetores densos de 768 dimensões. O Transformer não é treinado; somente as árvores aprendem fronteiras de decisão.

Faremos uma busca moderada na validação. O custo reportado separa:

- extração de embeddings;
- treinamento do LightGBM;
- inferência do LightGBM.

Em produção, o custo real ponta a ponta inclui tradução e BERTimbau, mesmo que os embeddings tenham sido cacheados durante o experimento.

### Como o boosting transforma embeddings em classes?

Para $N$ exemplos:

$$
X_{emb}\in\mathbb{R}^{N\times768},
\qquad
y\in\{0,1,2\}^{N}.
$$

LightGBM constrói árvores sequencialmente. Cada nova árvore busca reduzir o erro residual/gradiente do conjunto atual:

$$
F_m(x)=F_{m-1}(x)+\eta f_m(x),
$$

onde $\eta$ é `learning_rate` e $f_m$ é a árvore adicionada na iteração $m$. No caso multiclasse, o modelo produz três escores e os converte em probabilidades.

| Parâmetro | Interpretação |
|---|---|
| `n_estimators` | número máximo de árvores/iterações |
| `learning_rate` | contribuição de cada nova árvore |
| `num_leaves` | complexidade máxima das partições |
| `max_depth` | limite explícito de profundidade |
| `reg_alpha` | regularização L1 |
| `reg_lambda` | regularização L2 |

Não há backpropagation do LightGBM para o BERTimbau:

```text
BERTimbau congelado ──produz──> arquivo de embeddings
LightGBM ──lê──> embeddings e rótulos de treino
```

Esse desacoplamento permite testar classificadores rapidamente depois de pagar uma vez o custo da extração. Para latência real, entretanto, a representação contextual continua sendo parte do pipeline.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. Máscaras booleanas ligam cada linha de embedding ao split persistido.
# 2. Cada configuração cria um LGBMClassifier independente com seed fixa.
# 3. Somente embeddings/rótulos de treino entram em `fit`.
# 4. O F1 Macro de validação determina o índice da configuração escolhida.
# 5. `predict_proba` no teste ocorre apenas para a configuração vencedora.
# 6. A extração BERTimbau não recebe gradientes do LightGBM.
# 7. `trainable_params` fica não aplicável porque árvores não têm pesos neurais comparáveis.
# 8. A configuração escolhida e seu F1 de validação são impressos para auditoria.
# Saída esperada: probabilidades, matriz de confusão e métricas por trilha.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Treinar LightGBM sobre embeddings congelados com seleção pela validação.
# -----------------------------------------------------------------------------
if EXECUTAR_LIGHTGBM:
    from lightgbm import LGBMClassifier

    split_masks = {
        name: (df["split"].to_numpy() == name)
        for name in ["train", "valid", "test"]
    }
    configs = [
        {"num_leaves": 15, "learning_rate": 0.05, "n_estimators": 300},
        {"num_leaves": 31, "learning_rate": 0.03, "n_estimators": 500},
        {"num_leaves": 31, "learning_rate": 0.05, "n_estimators": 300},
    ]
    if MODO_RAPIDO:
        configs = configs[:1]

    for track, X in embedding_cache.items():
        y = df["y"].map(LABEL2ID).to_numpy()
        fitted, scores = [], []
        for cfg in configs:
            clf = LGBMClassifier(
                **cfg, objective="multiclass", random_state=SEED,
                max_depth=-1, reg_alpha=0.1, reg_lambda=0.1,
                verbosity=-1
            )
            start = time.perf_counter()
            clf.fit(X[split_masks["train"]], y[split_masks["train"]])
            seconds = time.perf_counter() - start
            valid_pred = clf.predict(X[split_masks["valid"]])
            scores.append(f1_score(
                y[split_masks["valid"]], valid_pred, average="macro"
            ))
            fitted.append((clf, seconds))

        best = int(np.argmax(scores))
        clf, train_seconds = fitted[best]
        start = time.perf_counter()
        proba = clf.predict_proba(X[split_masks["test"]])
        infer_seconds = time.perf_counter() - start
        row, pred = evaluate_predictions(
            y[split_masks["test"]], proba,
            "BERTimbau embeddings + LightGBM", track,
            train_seconds, infer_seconds,
            translation_seconds=(
                TRANSLATION_TEST_SECONDS
                if track == "machine_translated_pt" else 0.0
            ),
            trainable_params=np.nan
        )
        all_results.append(row)
        all_test_predictions[("BERTimbau embeddings + LightGBM", track)] = pred
        print("Configuração escolhida:", configs[best], "F1 valid:", scores[best])

## 21. BERTimbau + LoRA

**Pergunta de aprendizagem:** podemos adaptar a atenção ao domínio financeiro atualizando poucos parâmetros?

Configuração principal:

- `r=8`;
- `lora_alpha=16`;
- `lora_dropout=0.10`;
- alvos `query` e `value`;
- AdamW, warmup e early stopping;
- backbone `neuralmind/bert-base-portuguese-cased`.

O `TaskType.SEQ_CLS` garante que a cabeça de classificação permaneça treinável. Antes do treino, imprimimos a fração treinável:

$$
\rho=100\times
\frac{\#\text{parâmetros treináveis}}
{\#\text{parâmetros totais}}.
$$

### Arquitetura, dimensões e gradientes no LoRA

Fluxo para batch $N$ e sequência $L$:

```text
input_ids + attention_mask
(N, L)
    ↓ BERTimbau com pesos originais congelados
representações contextuais
(N, L, 768)
    ↓ vetor agregado da sequência
(N, 768)
    ↓ cabeça classificadora treinável
(N, 3 logits)
    ↓ CrossEntropyLoss
um escalar
```

Em uma projeção original $W_Q\in\mathbb{R}^{768\times768}$, fine-tuning completo atualizaria $589.824$ pesos. Com $r=8$, LoRA aprende aproximadamente:

$$
\#A+\#B
=8\times768+768\times8
=12.288
$$

pesos por projeção adaptada, antes de considerar camadas e vieses. A atualização aplicada no forward é:

$$
h=W_0x+\frac{\alpha}{r}BAx.
$$

`W_0` participa do cálculo, mas não recebe gradiente atualizável. `A`, `B` e a cabeça recebem gradientes.

#### Por que Query e Value?

- `Query` altera o que cada token procura;
- `Value` altera a informação propagada;
- `Key` permanece congelada nesta configuração.

Não comparamos vários alvos porque a especificação prioriza uma metodologia representativa.

#### Otimização

AdamW separa a atualização do weight decay. Warmup aumenta gradualmente a taxa de aprendizado nas primeiras etapas:

$$
\eta_t=\eta_{\max}\frac{t}{T_{warmup}},
\quad t\leq T_{warmup}.
$$

Depois, o scheduler padrão do Trainer reduz a taxa. Early stopping observa `eval_f1_macro`, restaura o checkpoint de maior valor e impede que o teste determine a época.

| Componente | Estado |
|---|---|
| embeddings originais | congelados |
| blocos originais do BERTimbau | congelados |
| matrizes LoRA em Query/Value | treináveis |
| cabeça de três classes | treinável |
| tokenizer | fixo |

A célula trata renomeações recentes da API (`eval_strategy` e `processing_class`) por inspeção de assinatura, deixando explícito qual argumento a versão instalada aceita.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. Cada DataFrame vira Dataset Hugging Face mantendo apenas texto e label.
# 2. `map(batched=True)` tokeniza eficientemente e remove a string após produzir IDs.
# 3. O modelo base recebe mapeamentos de classe explícitos.
# 4. `LoraConfig` adapta Query/Value com rank 8; os pesos originais continuam congelados.
# 5. `print_trainable_parameters` comprova a fração realmente atualizada.
# 6. `compute_metrics` usa logits da validação e retorna a chave monitorada.
# 7. A inspeção de assinatura trata nomes de argumentos entre versões do Transformers.
# 8. Early stopping e `load_best_model_at_end` restauram a melhor época.
# 9. O teste só entra em `trainer.predict` depois de `trainer.train`.
# 10. Softmax estável converte os logits finais em probabilidades comparáveis.
# Saída esperada: adapter LoRA, probabilidades de teste e métricas.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Tokenizar datasets, aplicar LoRA e treinar com seleção pela validação.
# -----------------------------------------------------------------------------
def run_lora_track(text_col, track_name):
    from datasets import Dataset
    from peft import LoraConfig, TaskType, get_peft_model
    from transformers import (
        AutoModelForSequenceClassification, AutoTokenizer,
        DataCollatorWithPadding, EarlyStoppingCallback,
        Trainer, TrainingArguments
    )

    tokenizer = AutoTokenizer.from_pretrained(BERTIMBAU_MODEL)

    def make_hf(frame):
        ds = Dataset.from_dict({
            "text": frame[text_col].tolist(),
            "labels": frame["label_id"].tolist(),
        })
        return ds.map(
            lambda batch: tokenizer(
                batch["text"], truncation=True, max_length=MAX_LENGTH_BERT
            ),
            batched=True, remove_columns=["text"]
        )

    train_ds, valid_ds, test_ds = (
        make_hf(train_df), make_hf(valid_df), make_hf(test_df)
    )
    base = AutoModelForSequenceClassification.from_pretrained(
        BERTIMBAU_MODEL, num_labels=3,
        id2label=ID2LABEL, label2id=LABEL2ID
    )
    config = LoraConfig(
        task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16,
        lora_dropout=0.10, target_modules=["query", "value"],
        bias="none"
    )
    model = get_peft_model(base, config)
    model.print_trainable_parameters()

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        pred = np.argmax(logits, axis=1)
        return {"f1_macro": f1_score(labels, pred, average="macro")}

    output_dir = MODEL_DIR / f"bertimbau_lora_{track_name}"
    # Transformers renomeou evaluation_strategy para eval_strategy e tokenizer
    # para processing_class em versões recentes. A inspeção mantém o notebook
    # didático e compatível sem esconder qual opção está sendo usada.
    import inspect
    argument_names = inspect.signature(TrainingArguments.__init__).parameters
    training_kwargs = dict(
        output_dir=str(output_dir),
        learning_rate=2e-4,
        per_device_train_batch_size=8 if MODO_RAPIDO else 16,
        per_device_eval_batch_size=16 if MODO_RAPIDO else 32,
        num_train_epochs=1 if MODO_RAPIDO else 8,
        weight_decay=0.01,
        warmup_ratio=0.10,
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=SEED,
    )
    strategy_name = (
        "eval_strategy" if "eval_strategy" in argument_names
        else "evaluation_strategy"
    )
    training_kwargs[strategy_name] = "epoch"
    args = TrainingArguments(**training_kwargs)

    trainer_kwargs = dict(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=valid_ds,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainer_argument_names = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in trainer_argument_names:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer
    trainer = Trainer(**trainer_kwargs)
    start = time.perf_counter()
    trainer.train()
    train_seconds = time.perf_counter() - start
    trainer.save_model(str(output_dir / "best_adapter"))

    start = time.perf_counter()
    prediction = trainer.predict(test_ds)
    infer_seconds = time.perf_counter() - start
    logits = prediction.predictions
    exp = np.exp(logits - logits.max(axis=1, keepdims=True))
    proba = exp / exp.sum(axis=1, keepdims=True)
    row, pred = evaluate_predictions(
        test_df["label_id"], proba, "BERTimbau + LoRA", track_name,
        train_seconds, infer_seconds,
        translation_seconds=(
            TRANSLATION_TEST_SECONDS
            if track_name == "machine_translated_pt" else 0.0
        ),
        trainable_params=count_trainable_parameters(model)
    )
    return trainer, row, pred, proba


if EXECUTAR_LORA:
    lora_artifacts = {}
    for text_col, track in [
        ("text_pt", "reference_pt"),
        ("text_pt_mt", "machine_translated_pt"),
    ]:
        trainer, row, pred, proba = run_lora_track(text_col, track)
        lora_artifacts[track] = trainer
        all_results.append(row)
        all_test_predictions[("BERTimbau + LoRA", track)] = pred

### Checkpoint conceitual 3

- TF-IDF aprende pesos supervisionados sobre estatísticas lexicais.
- BiLSTM aprende a representação do zero e depende fortemente do tamanho do corpus.
- BERTimbau congelado oferece representação contextual sem adaptação.
- LightGBM aprende apenas a fronteira sobre os embeddings.
- LoRA altera o comportamento do Transformer com matrizes de baixo posto.
- O melhor modelo não é necessariamente o melhor sistema: tradução e latência também importam.

## 22. Análise de erros

Uma métrica média não explica **por que** o sistema falha. Construiremos uma tabela por exemplo com:

- original inglês;
- referência e tradução automática;
- classe verdadeira;
- previsão de cada modelo;
- números e negações;
- concordância entre modelos;
- mudança da previsão entre as duas trilhas.

Categorias qualitativas sugeridas:

1. negação;
2. ambiguidade;
3. números e percentuais;
4. termos financeiros;
5. omissão ou adição na tradução;
6. confusão `neutral ↔ positive` ou `neutral ↔ negative`.

### Decompondo o erro ponta a ponta

Defina:

- $\hat{y}_{ref}$: previsão usando a tradução de referência;
- $\hat{y}_{mt}$: previsão usando a tradução automática;
- $y$: rótulo verdadeiro.

Quatro situações são especialmente úteis:

| Situação | Interpretação provável |
|---|---|
| $\hat{y}_{ref}=y$ e $\hat{y}_{mt}\neq y$ | tradução pode ter degradado pistas |
| $\hat{y}_{ref}\neq y$ e $\hat{y}_{mt}=y$ | tradução pode ter simplificado ou favorecido o modelo |
| ambas corretas | exemplo robusto |
| ambas erradas | ambiguidade, rótulo ou limitação do classificador |

A taxa de mudança de previsão é:

$$
R_{mudança}
=\frac{1}{N}\sum_{i=1}^{N}
\mathbb{1}[\hat{y}^{ref}_i\neq\hat{y}^{mt}_i].
$$

Ela mede sensibilidade, não qualidade: uma mudança pode corrigir ou introduzir erro. Por isso contamos separadamente `reference_correct_mt_wrong` e `reference_wrong_mt_correct`.

Uma análise responsável seleciona casos por regras reproduzíveis — negação, presença de número, discordância ou quantidade de acertos — em vez de escolher apenas exemplos que sustentem uma narrativa.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. A tabela começa com exemplos do teste na mesma ordem usada pelos preditores.
# 2. Números e negações viram flags para ordenar análises temáticas.
# 3. Nomes de modelos são higienizados apenas para formar nomes de coluna válidos.
# 4. Cada vetor de IDs previstos é convertido novamente em rótulo textual.
# 5. `n_correct` soma acertos por exemplo entre todas as execuções disponíveis.
# 6. A ordenação coloca primeiro erros compartilhados com fenômenos linguísticos relevantes.
# 7. Se nenhum modelo rodou, a célula explica por que não há tabela.
# Saída esperada: dataset de erros por exemplo, modelo e trilha.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Montar uma tabela de erros alinhada pelo example_id, sem escolher exemplos à mão.
# -----------------------------------------------------------------------------
error_table = test_df[[
    "example_id", "text_en", "text_pt", "text_pt_mt", "y", "label_id"
]].copy()
error_table["has_number"] = error_table["text_pt_mt"].map(
    lambda x: bool(extract_numbers(x))
)
error_table["has_negation"] = error_table["text_pt_mt"].map(has_negation_pt)

for (model_name, track), pred in all_test_predictions.items():
    safe_name = re.sub(r"[^a-z0-9]+", "_", model_name.lower()).strip("_")
    col = f"pred__{safe_name}__{track}"
    error_table[col] = [ID2LABEL[int(i)] for i in pred]

prediction_cols = [c for c in error_table if c.startswith("pred__")]
if prediction_cols:
    error_table["n_correct"] = sum(
        error_table[c].eq(error_table["y"]).astype(int) for c in prediction_cols
    )
    display(error_table.sort_values(
        ["n_correct", "has_negation", "has_number"],
        ascending=[True, False, False]
    ).head(30))
else:
    print("Execute ao menos um modelo para preencher a análise de erros.")

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. Identificamos modelos que possuem previsões nas duas trilhas.
# 2. Comparações usam arrays alinhados pela ordem imutável de `test_df`.
# 3. `changed` marca qualquer troca de classe, correta ou incorreta.
# 4. Os dois contadores direcionais distinguem degradação de correção após tradução.
# 5. Nenhum exemplo é selecionado manualmente e nenhuma métrica ajusta o modelo.
# Saída esperada: taxa de sensibilidade à tradução para cada arquitetura.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Comparar a mudança de previsão entre referência e tradução para cada modelo.
# -----------------------------------------------------------------------------
paired_rows = []
model_names = sorted({key[0] for key in all_test_predictions})
for model_name in model_names:
    key_ref = (model_name, "reference_pt")
    key_mt = (model_name, "machine_translated_pt")
    if key_ref not in all_test_predictions or key_mt not in all_test_predictions:
        continue
    ref_pred = all_test_predictions[key_ref]
    mt_pred = all_test_predictions[key_mt]
    changed = ref_pred != mt_pred
    paired_rows.append({
        "model": model_name,
        "prediction_change_rate": changed.mean(),
        "n_changed": changed.sum(),
        "reference_correct_mt_wrong": (
            (ref_pred == test_df["label_id"].to_numpy())
            & (mt_pred != test_df["label_id"].to_numpy())
        ).sum(),
        "reference_wrong_mt_correct": (
            (ref_pred != test_df["label_id"].to_numpy())
            & (mt_pred == test_df["label_id"].to_numpy())
        ).sum(),
    })

display(pd.DataFrame(paired_rows))

## 23. Comparação consolidada e trade-offs

A tabela final deve responder:

- qual modelo teve maior F1 Macro?
- qual perdeu menos desempenho com tradução automática?
- qual treinou e inferiu mais rapidamente?
- qual é mais interpretável?
- quanto custa o pipeline completo?

Uma comparação útil não reduz tudo a um único número:

| Abordagem | Força provável | Limitação provável |
|---|---|---|
| TF-IDF + Logística | rapidez e coeficientes interpretáveis | contexto limitado |
| BiLSTM + Optuna | sequência aprendida do domínio | dados e treino do zero |
| BERTimbau + LightGBM | representação forte e reutilizável | extração contextual custa |
| BERTimbau + LoRA | adaptação contextual eficiente | maior complexidade operacional |

O melhor custo-benefício depende da restrição de latência, memória, explicabilidade e frequência de atualização.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. Resultados acumulados são ordenados por F1 Macro e persistidos em CSV.
# 2. `style.format` altera somente a exibição, nunca os valores armazenados.
# 3. O pivot coloca as duas trilhas lado a lado por modelo.
# 4. Delta positivo significa que a tradução automática superou a referência naquele modelo.
# 5. O gráfico usa eixo comum de zero a um para evitar exagero visual.
# 6. `savefig` ocorre antes de `show`, garantindo que o PNG seja materializado.
# 7. A célula continua válida quando somente parte dos experimentos foi executada.
# Saída esperada: ranking, deltas, tabela persistida e gráfico comparativo.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Consolidar métricas, calcular delta da tradução e salvar o relatório.
# -----------------------------------------------------------------------------
if all_results:
    results = save_results(all_results)
    display(results.style.format({
        "accuracy": "{:.4f}", "precision_macro": "{:.4f}",
        "recall_macro": "{:.4f}", "f1_macro": "{:.4f}",
        "log_loss": "{:.4f}", "train_seconds": "{:.2f}",
        "inference_seconds": "{:.2f}", "pipeline_seconds": "{:.2f}",
    }))

    paired = results.pivot_table(
        index="model", columns="input_track", values="f1_macro", aggfunc="first"
    )
    if {"reference_pt", "machine_translated_pt"}.issubset(paired.columns):
        paired["delta_f1_mt_minus_reference"] = (
            paired["machine_translated_pt"] - paired["reference_pt"]
        )
    display(paired.sort_values(
        "machine_translated_pt", ascending=False
    ) if "machine_translated_pt" in paired else paired)

    plt.figure(figsize=(11, 5))
    sns.barplot(
        data=results, x="f1_macro", y="model", hue="input_track",
        orient="h"
    )
    plt.xlim(0, 1)
    plt.title("F1 Macro por modelo e origem do texto")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "f1_macro_comparison.png", dpi=160)
    plt.show()
else:
    print("Nenhum experimento foi executado nesta sessão.")

## 24. Conclusões orientadas por evidência

Preencha esta seção somente após executar os quatro experimentos.

### Perguntas finais

1. Qual abordagem apresentou o maior F1 Macro em `text_pt_mt`?
2. Qual apresentou a menor queda em relação a `text_pt`?
3. Quais classes foram mais afetadas pela tradução?
4. Os erros vieram de negações, números, ambiguidade ou terminologia?
5. Qual abordagem oferece o melhor custo-benefício?
6. Quando a interpretabilidade do TF-IDF vale mais que o ganho de um Transformer?
7. O custo do tradutor domina ou não a latência ponta a ponta?

### Limitações que devem ser reconhecidas

- uma única base e um único domínio;
- tradução portuguesa de referência não é necessariamente uma tradução perfeita;
- a anotação de sentimento pode ser ambígua;
- tempos dependem do hardware e do cache;
- Optuna otimiza apenas a BiLSTM;
- não há fine-tuning completo nem comparação entre múltiplos tradutores;
- significância estatística exige repetição com diferentes sementes ou bootstrap.

### Síntese

O benchmark foi desenhado para tornar visível a cadeia causal:

$$
\text{qualidade da tradução}
\rightarrow
\text{representação}
\rightarrow
\text{fronteira de decisão}
\rightarrow
\text{erro final}.
$$

Mais do que eleger um vencedor, o projeto ensina como medir cada escolha sem vazamento e como relacionar desempenho, custo e interpretabilidade.

In [ ]:
# =============================================================================
# LEITURA GUIADA DA CÉLULA
# =============================================================================
# Leitura guiada:
# 1. O manifesto registra decisões que definem a identidade do experimento.
# 2. Caminhos são convertidos para string para serialização JSON.
# 3. `ensure_ascii=False` preserva caracteres portugueses de forma legível.
# 4. O arquivo não contém modelos ou dados; ele aponta para seus artefatos.
# 5. A tabela exibida facilita uma última conferência no notebook.
# Saída esperada: `experiment_manifest.json` reproduzível e legível.
# =============================================================================

# -----------------------------------------------------------------------------
# Objetivo desta célula:
# Registrar versões finais e caminhos dos artefatos para auditoria.
# -----------------------------------------------------------------------------
manifest = {
    "seed": SEED,
    "data_path": str(DATA_PATH),
    "split_path": str(SPLIT_PATH),
    "translation_path": str(TRANSLATION_PATH),
    "translation_model": TRANSLATION_MODEL,
    "bertimbau_model": BERTIMBAU_MODEL,
    "labels": LABELS,
    "device": str(DEVICE),
    "results_path": str(RESULTS_PATH),
}
(REPORT_DIR / "experiment_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
display(pd.Series(manifest, name="valor").to_frame())